In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch02-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "02-06-reaction-rules.py"
work_dir = Path(work.name)


# 用反應規則產生候選產物

先執行環境設定，再執行寫入程式與分析結果兩格。修改計算條件後，請重新執行後面兩格。輸出暫存在本次工作資料夾。

<!-- concept-ch00-05:start -->
座標變化和反應轉換可以用同一組問題來區分。

| 比較項目 | 旋轉正丁醇單鍵 | 把醇轉成羰基候選 |
|---|---|---|
| 原子如何相連 | 保留原有連接 | C–O 鍵級改變 |
| 三維座標 | 可產生不同構形 | 若要做三維計算，須另外產生候選產物的座標 |
| 可沿用的量測 | 仍要看量測對象與條件 | 不可把反應物 logS 當產物值 |
| 本節輸出代表什麼 | 起始構形及力場最佳化結果 | 符合局部規則的候選結構 |

比較上表可見：產生構形時，原子連接方式與鍵的種類不變；套用本節的反應規則時，碳氧單鍵會改成雙鍵。兩者都不能直接當成實驗已觀察到的結果。
<!-- concept-ch00-05:end -->


In [ ]:
%%writefile "{script}"
from pathlib import Path
import os, json
os.environ["MPLBACKEND"] = "Agg"
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolTransforms
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
_, parts, _ = dc.molnet.load_delaney(featurizer=dc.feat.RawFeaturizer(smiles=True), splitter='index', transformers=[], reload=False, data_dir=os.environ.get('DEEPCHEM_DATA_DIR'))
rows=[]
for part in parts:
 for smi in part.ids:
  mol=Chem.MolFromSmiles(smi)
  assert mol is not None
  rows.append((len(rows)+1,str(smi),mol))

assert len(rows) == 1128
reaction_smarts = "[C;X4;H1,H2;+0:1]-[O;X2;H1;+0:2]>>[C:1]=[O:2]"
rxn = AllChem.ReactionFromSmarts(reaction_smarts)

def products_for(mol):
    products = {}
    for bundle in rxn.RunReactants((mol,)):
        for product in bundle:
            Chem.SanitizeMol(product)
            smiles = Chem.MolToSmiles(product)
            products[smiles] = product
    return products
# 概念示意 — 實際應用請使用下方 DeepChem 版本
entries=[]
for name,smi in [('Ethanol','CCO'),('Propan-2-ol','CC(O)C'),('tert-Butanol','CC(C)(C)O'),('Phenol','Oc1ccccc1')]:
 reactant=Chem.MolFromSmiles(smi)
 products=products_for(reactant)
 print(name,'concept unique candidates',list(products))
 entries.append((name,reactant,products))
assert list(entries[0][2])==['CC=O']
assert list(entries[1][2])==['CC(C)=O']
assert not entries[2][2] and not entries[3][2]
# DeepChem 版本：對每筆 ESOL 結構套用規則，保留原資料列編號。
matched=[]
for rid,raw,reactant in rows:
 candidates=products_for(reactant)
 if candidates:
  matched.append((rid,raw,sorted(candidates)))
print('ESOL matched rows',len(matched))
assert any(rid==984 and products==['CC=O'] for rid,raw,products in matched)

records = []
for rid, raw, candidates in matched:
    for candidate in candidates:
        records.append({"source_row": rid, "original_smiles": raw, "candidate_smiles": candidate})
candidate_table = pd.DataFrame(records)
candidate_table.to_csv("reaction_candidates.csv", index=False)
Path("results.json").write_text(json.dumps({"dataset_rows": len(rows), "matched_rows": len(matched),
    "candidate_rows": len(records), "concept_examples": {name: list(products) for name, _, products in entries}}, indent=2))
out = Path('figures')
out.mkdir(exist_ok=True)
draw_mols=[];legends=[]
for name,reactant,products in entries:
 draw_mols.extend([reactant,next(iter(products.values()),None)])
 legends.extend([name,'Candidate' if products else 'No match for this rule'])
panel=Draw.MolsToGridImage(draw_mols,molsPerRow=2,subImgSize=(350,180),legends=legends)
from PIL import ImageDraw, ImageFont
canvas=ImageDraw.Draw(panel)
font=ImageFont.load_default(size=18)
for row,(_,_,products) in enumerate(entries):
 if not products:
  canvas.text((525,row*180+155),'No match for this rule',font=font,fill='black',anchor='mm')
panel.save(out/'02-06-reaction-candidates.png')


In [ ]:
from IPython.display import Image, display
import pandas as pd

run = subprocess.run([python, str(script)], cwd=work_dir, capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
candidate_table = pd.read_csv(work_dir / "reaction_candidates.csv")
display(candidate_table)
display(Image(filename=str(work_dir / "figures" / "02-06-reaction-candidates.png")))
